# PHASE 1 : DATA GENERATION   

In [0]:
%pip install faker

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import os
import random
import pandas as pd
from faker import Faker
from datetime import datetime

In [0]:
fake = Faker("en_IN")
Faker.seed(42)
random.seed(42)

In [0]:
import os
output_dir = "data/raw"
os.makedirs(output_dir, exist_ok=True)
print("Output folder:", os.path.abspath(output_dir))

Output folder: /Workspace/Users/daniharshit0@gmail.com/data/raw


In [0]:
NUM_CUSTOMERS = 1000
NUM_PRODUCTS = 200
NUM_ORDERS = 5000

print("Dataset configuration loaded.")

Dataset configuration loaded.


In [0]:
categories = {
    "Electronics": {
        "price_range": (500, 100000),
        "subcategories": [
            "Mobile",
            "Laptop",
            "Accessories",
            "Audio",
            "Camera"
        ]
    },

    "Clothing": {
        "price_range": (300, 5000),
        "subcategories": [
            "Men",
            "Women",
            "Kids"
        ]
    },

    "Books": {
        "price_range": (100, 1500),
        "subcategories": [
            "Programming",
            "Fiction",
            "Education",
            "Business"
        ]
    },

    "Sports": {
        "price_range": (500, 10000),
        "subcategories": [
            "Indoor",
            "Outdoor",
            "Fitness"
        ]
    },

    "Beauty": {
        "price_range": (100, 5000),
        "subcategories": [
            "Skin Care",
            "Hair Care",
            "Cosmetics"
        ]
    },

    "Home": {
        "price_range": (500, 30000),
        "subcategories": [
            "Furniture",
            "Kitchen",
            "Decor"
        ]
    }
}

print("Categories loaded successfully!")

Categories loaded successfully!


### Generating customers.csv

In [0]:
customer_types = ["REGULAR", "PREMIUM", "VIP"]
customers = []

for customer_id in range(1, NUM_CUSTOMERS + 1):

    customers.append({
        "customer_id": customer_id,
        "customer_name": fake.name(),
        "email": fake.unique.email(),
        "registration_date": fake.date_between(
            start_date="-3y",
            end_date="today"
        ),
        "customer_type": random.choices(
            customer_types,
            weights=[70, 20, 10],
            k=1
        )[0]
    })

customers_df = pd.DataFrame(customers)

# Exactly 2% invalid emails
invalid_indices = random.sample(
    list(customers_df.index),
    k=int(NUM_CUSTOMERS * 0.02)
)

for idx in invalid_indices:

    email = customers_df.at[idx, "email"]

    username = email.split("@")[0]

    invalid_type = random.choice([1, 2, 3])

    if invalid_type == 1:
        # Remove @
        invalid_email = email.replace("@", "")

    elif invalid_type == 2:
        # Missing domain
        invalid_email = username + "@"

    else:
        # Missing @
        invalid_email = username + ".com"

    customers_df.at[idx, "email"] = invalid_email

    customers_df.to_csv(
    os.path.join(output_dir, "customers.csv"),
    index=False
)

print("customers.csv saved successfully!")

customers.csv saved successfully!


### Generating products.csv

In [0]:
product_names = {
    "Electronics": [
        "Laptop", "Smartphone", "Keyboard", "Mouse",
        "Monitor", "Tablet", "Camera", "Speaker"
    ],
    "Clothing": [
        "T-Shirt", "Jeans", "Jacket", "Shoes",
        "Sweater", "Dress", "Kurta"
    ],
    "Books": [
        "Python Guide", "SQL Handbook", "AI Basics",
        "Algorithms", "Machine Learning", "Java Programming"
    ],
    "Sports": [
        "Football", "Cricket Bat", "Yoga Mat",
        "Basketball", "Dumbbells", "Tennis Racket"
    ],
    "Beauty": [
        "Perfume", "Shampoo", "Face Wash",
        "Lipstick", "Body Lotion"
    ],
    "Home": [
        "Chair", "Table", "Lamp",
        "Sofa", "Microwave", "Mixer"
    ]
}

products = []

for product_id in range(1, NUM_PRODUCTS + 1):

    category = random.choice(list(categories.keys()))

    subcategory = random.choice(
        categories[category]["subcategories"]
    )

    min_price, max_price = categories[category]["price_range"]

    product_name = random.choice(product_names[category])

    # ---------- Intentional Noise ----------

    # 5% extra spaces
    if random.random() < 0.05:
        product_name = "  " + product_name + "   "

    # 5% mixed case
    elif random.random() < 0.05:
        product_name = "".join(
            c.upper() if random.random() > 0.5 else c.lower()
            for c in product_name
        )

    products.append({

        "product_id": product_id,

        "product_name": product_name,

        "category": category,

        "subcategory": subcategory,

        "cost_price": round(
            random.uniform(min_price, max_price),
            2
        )

    })

products_df = pd.DataFrame(products)
products_df.to_csv(
    os.path.join(output_dir, "products.csv"),
    index=False
)

print("products.csv saved successfully!")
products_df.head()

products.csv saved successfully!


,product_id,product_name,category,subcategory,cost_price
0,1,SQL Handbook,Books,Fiction,253.90
1,2,Laptop,Electronics,Laptop,59090.87
2,3,Monitor,Electronics,Laptop,52845.51
3,4,Sofa,Home,Furniture,18013.15
4,5,CRIcKet BaT,Sports,Fitness,1198.16


### Generating orders.csv 

In [0]:
statuses = [
    "PLACED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

# 50 unique 6-digit region codes
region_codes = random.sample(range(100000, 999999), 50)

orders = []

for order_id in range(1, NUM_ORDERS + 1):

    # Select a random customer
    customer = customers_df.sample(1).iloc[0]

    # Generate order datetime
    order_datetime = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    orders.append({

        "order_id": order_id,

        "customer_id": customer["customer_id"],

        "order_date": order_datetime.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

        "status": random.choice(statuses),

        "region_code": random.choice(region_codes)

    })

orders_df = pd.DataFrame(orders)

print("Orders generated successfully!")

null_indices = random.sample(
    list(orders_df.index),
    k=int(NUM_ORDERS * 0.05)
)

orders_df.loc[null_indices, "customer_id"] = None

available_indices = list(
    orders_df.index.difference(null_indices)
)

wrong_date_indices = random.sample(
    available_indices,
    k=int(NUM_ORDERS * 0.05)
)

for idx in wrong_date_indices:

    dt = pd.to_datetime(
        orders_df.at[idx, "order_date"]
    )

    orders_df.at[idx, "order_date"] = dt.strftime(
        "%d-%m-%Y %H:%M:%S"
    )

    orders_df.to_csv(
    os.path.join(output_dir, "orders.csv"),
    index=False
)

print("orders.csv saved successfully!")
orders_df.head()

Orders generated successfully!
orders.csv saved successfully!


,order_id,customer_id,order_date,status,region_code
0,1,320.0,2025-07-16 17:43:39,RETURNED,158840
1,2,460.0,2025-06-03 22:24:42,DELIVERED,471658
2,3,847.0,2026-03-19 21:02:03,PLACED,999636
3,4,369.0,2025-03-28 18:41:52,DELIVERED,279708
4,5,501.0,2025-05-15 12:23:29,RETURNED,693343


### Generating order_items.csv

In [0]:
order_items = []

item_id = 1

for _, order in orders_df.iterrows():

    # Each order contains between 1 and 5 products
    num_products = random.randint(1, 5)

    # Choose unique products
    selected_products = products_df.sample(
        n=num_products,
        replace=False
    )

    for _, product in selected_products.iterrows():

        order_items.append({

            "item_id": item_id,

            "order_id": order["order_id"],

            "product_id": product["product_id"],

            "quantity": random.randint(1, 5),

            "unit_price": product["cost_price"],

            "discount_percent": random.randint(0, 100)

        })

        item_id += 1

order_items_df = pd.DataFrame(order_items)

print("Order Items generated successfully!")

negative_indices = random.sample(
    list(order_items_df.index),
    k=int(len(order_items_df) * 0.03)
)

order_items_df.loc[
    negative_indices,
    "quantity"
] *= -1

order_items_df.to_csv(
    os.path.join(output_dir, "order_items.csv"),
    index=False
)

print("order_items.csv saved successfully!")
order_items_df.head()

Order Items generated successfully!
order_items.csv saved successfully!


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,1,81,5,641.30,18
1,2,1,3,5,52845.51,94
2,3,2,15,2,17548.60,28
3,4,2,171,2,5623.35,38
4,5,2,37,3,3250.66,92


#  PHASE 2 : DATA CLEANING 

In [0]:
import pandas as pd
import os

raw_path = "data/raw"
clean_path = "data/cleaned"

os.makedirs(clean_path, exist_ok=True)

customers = pd.read_csv(os.path.join(raw_path, "customers.csv"))
products = pd.read_csv(os.path.join(raw_path, "products.csv"))
orders = pd.read_csv(os.path.join(raw_path, "orders.csv"))
order_items = pd.read_csv(os.path.join(raw_path, "order_items.csv"))

print(customers.shape)
print(products.shape)
print(orders.shape)
print(order_items.shape)

(1000, 5)
(200, 5)
(5000, 5)
(15109, 6)



### Generating orders_clean.csv

In [0]:
def clean_orders(orders_df, customers_df):

    orders_clean = orders_df.copy()
    # 1. Fix date formats
    orders_clean["order_date"] = pd.to_datetime(
        orders_clean["order_date"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    orders_clean["order_date"] = orders_clean["order_date"].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    # 2. Handle NULL customer IDs
    valid_customer_ids = customers_df["customer_id"].tolist()

    replacement_ids = random.choices(
        valid_customer_ids,
        k=orders_clean["customer_id"].isna().sum()
    )

    orders_clean.loc[
        orders_clean["customer_id"].isna(),
        "customer_id"
    ] = replacement_ids

    orders_clean["customer_id"] = orders_clean["customer_id"].astype(int)

    return orders_clean

In [0]:
orders_clean = clean_orders(
    orders,
    customers
)
orders_clean.to_csv(
    os.path.join(clean_path, "orders_clean.csv"),
    index=False
)

print("orders_clean.csv saved successfully!")
orders_clean.head()



orders_clean.csv saved successfully!


,order_id,customer_id,order_date,status,region_code
0,1,320,2025-07-16 17:43:39,RETURNED,158840
1,2,460,2025-06-03 22:24:42,DELIVERED,471658
2,3,847,2026-03-19 21:02:03,PLACED,999636
3,4,369,2025-03-28 18:41:52,DELIVERED,279708
4,5,501,2025-05-15 12:23:29,RETURNED,693343


### Generating products_clean.csv

In [0]:
print("Remaining NULL customer IDs :")

print(
    orders_clean["customer_id"].isnull().sum()
)

print()

print("Sample dates:")

print(
    orders_clean["order_date"].head()
)

Remaining NULL customer IDs :
0

Sample dates:
0    2025-07-16 17:43:39
1    2025-06-03 22:24:42
2    2026-03-19 21:02:03
3    2025-03-28 18:41:52
4    2025-05-15 12:23:29
Name: order_date, dtype: object


In [0]:
def clean_products(products_df):

    products_clean = products_df.copy()

    # Remove leading and trailing spaces
    products_clean["product_name"] = (
        products_clean["product_name"]
        .astype(str)
        .str.strip()
    )

    # Convert to Title Case
    products_clean["product_name"] = (
        products_clean["product_name"]
        .str.title()
    )

    return products_clean

In [0]:
products_clean = clean_products(products)
products_clean.to_csv(
    os.path.join(clean_path, "products_clean.csv"),
    index=False
)

print("products_clean.csv saved successfully!")
products_clean.head(10)

products_clean.csv saved successfully!


,product_id,product_name,category,subcategory,cost_price
0,1,Sql Handbook,Books,Fiction,253.90
1,2,Laptop,Electronics,Laptop,59090.87
2,3,Monitor,Electronics,Laptop,52845.51
3,4,Sofa,Home,Furniture,18013.15
4,5,Cricket Bat,Sports,Fitness,1198.16
5,6,Keyboard,Electronics,Accessories,63491.13
6,7,Face Wash,Beauty,Cosmetics,1545.00
7,8,Lipstick,Beauty,Cosmetics,4278.68
8,9,Mixer,Home,Decor,26698.69
9,10,Basketball,Sports,Outdoor,3469.21


### Validation

In [0]:
import re

def validate_emails(customers_df):

    customers_copy = customers_df.copy()

    email_pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    invalid_customers = customers_copy[
        ~customers_copy["email"].str.match(email_pattern, na=False)
    ]

    return invalid_customers

In [0]:
invalid_emails = validate_emails(customers)

print("Invalid Emails Found:", len(invalid_emails))

invalid_emails.head()

Invalid Emails Found: 20


,customer_id,customer_name,email,registration_date,customer_type
62,63,Michael Kala,sachdevanikitaexample.com,2025-03-25,REGULAR
93,94,Pavani Contractor,manthanlataexample.org,2023-10-26,REGULAR
101,102,Turvi Sachdev,guhazarna.com,2025-06-21,PREMIUM
125,126,Ishwar Sidhu,dhaliwalupkaarexample.net,2025-10-10,REGULAR
194,195,Harinakshi Grover,xtankexample.net,2024-05-27,REGULAR


In [0]:
def check_referential_integrity(order_items_df, orders_df):

    # Existing order IDs
    valid_order_ids = set(orders_df["order_id"])

    # Find invalid references
    invalid_order_items = order_items_df[
        ~order_items_df["order_id"].isin(valid_order_ids)
    ]

    return invalid_order_items

In [0]:
invalid_order_items = check_referential_integrity(
    order_items,
    orders
)

print("Invalid Order References :", len(invalid_order_items))

invalid_order_items.head()

Invalid Order References : 0


,item_id,order_id,product_id,quantity,unit_price,discount_percent


### Generating customers_clean.csv

In [0]:
customers.to_csv(
    os.path.join(clean_path, "customers_clean.csv"),
    index=False
)

print("customers_clean.csv saved successfully!")

customers_clean.csv saved successfully!


### Generating order_items_clean.csv

In [0]:
order_items.to_csv(
    os.path.join(clean_path, "order_items_clean.csv"),
    index=False
)

print("order_items_clean.csv saved successfully!")

order_items_clean.csv saved successfully!


In [0]:
import pandas as pd
import os

clean_path = "data/cleaned"

customers_clean = pd.read_csv(
    os.path.join(clean_path, "customers_clean.csv")
)

products_clean = pd.read_csv(
    os.path.join(clean_path, "products_clean.csv")
)

orders_clean = pd.read_csv(
    os.path.join(clean_path, "orders_clean.csv")
)

order_items_clean = pd.read_csv(
    os.path.join(clean_path, "order_items_clean.csv")
)

print("Cleaned datasets loaded successfully!")

print("Customers:", customers_clean.shape)
print("Products:", products_clean.shape)
print("Orders:", orders_clean.shape)
print("Order Items:", order_items_clean.shape)

Cleaned datasets loaded successfully!
Customers: (1000, 5)
Products: (200, 5)
Orders: (5000, 5)
Order Items: (15109, 6)


In [0]:
# Save cleaned datasets as permanent Databricks tables

spark.createDataFrame(customers_clean) \
    .write \
    .mode("overwrite") \
    .saveAsTable("customers")

spark.createDataFrame(products_clean) \
    .write \
    .mode("overwrite") \
    .saveAsTable("products")

spark.createDataFrame(orders_clean) \
    .write \
    .mode("overwrite") \
    .saveAsTable("orders")

spark.createDataFrame(order_items) \
    .write \
    .mode("overwrite") \
    .saveAsTable("order_items")

print("All cleaned datasets saved as Databricks tables!")

All cleaned datasets saved as Databricks tables!
